# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a structured, step-by-step template for loading and exploring a dataset using the `mlcroissant` library. We will reference all dataset entities using their unique `@id` as per FAIR and Croissant principles.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, their `@id`s, and field `@id`s.

In [ ]:
# List all record sets by @id and their fields
print("Available record sets and their fields (using @id):\n")
record_sets = list(ds.record_sets)
for recordset in record_sets:
    print(f"Record Set Name: {recordset.name}")
    print(f"  @id: {recordset.id}")
    print("  Fields:")
    for field in recordset.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print("")

## 3. Data Extraction
Load data from all available record sets into separate DataFrames. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set using their @id
# Build a list of record set @ids
record_set_ids = [rec.id for rec in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    recs = list(ds.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(recs)
    print(f"Loaded {len(recs)} records for Record Set: {record_set_id}")

# For illustration, print the columns and a preview for the first available record set
if len(record_set_ids) > 0:
    first_rs_id = record_set_ids[0]
    print(f"\nFields (@id) for Record Set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All fields are referenced by `@id` as extracted above.

In [ ]:
# Example: Assume first record set and choose a numeric field to explore (change as needed based on overview)
if len(record_set_ids) > 0:
    rs_id = first_rs_id
    df = dataframes[rs_id]
    # Attempt to select the first numeric field with at least 2 unique values
    numeric_field_id = None
    for col in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                if df[col].nunique() > 1:
                    numeric_field_id = col
                    break
        except Exception:
            continue
    if numeric_field_id is not None:
        print(f"Selected numeric field (@id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
        # Attempt grouping by another field if available
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() > 1:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean')
            print(f"Grouped filtered records by {group_field} (mean {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All field references use `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic visualization for the selected numeric field
if len(record_set_ids) > 0 and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    # If a group_field is available, show boxplot
    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Access and load a FAIR dataset described in Croissant format via its schema URL
- Explore available record sets and fields using their `@id`s
- Load and preview records from the dataset using the `mlcroissant` API
- Perform simple EDA such as filtering and normalization, referencing every entity by its unique `@id`
- Visualize field distributions and group statistics

This workflow shows how `mlcroissant` can streamline FAIR-aligned, reproducible data exploration in clinical medical research use cases.